# 22 — AI Project Manager
> Feed it raw update text — emails, call notes, Slack threads — and it keeps a living project record up to date: milestones, risks, blockers, ownership, and a green / amber / red status with an executive summary, all re-derived after every update.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


# ── Data model ────────────────────────────────────────────────────────────────

class Milestone(BaseModel):
    name: str = Field(description="Milestone name.")
    due_date: str = Field(description="Target completion date, e.g. 2024-Q3.")
    status: Literal["on_track", "at_risk", "delayed", "complete"] = Field(
        description="Current milestone status."
    )


class Risk(BaseModel):
    description: str = Field(description="Plain-English risk description.")
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="Estimated business impact if the risk materialises."
    )
    owner: str = Field(description="Person or team responsible for mitigation.")


class Blocker(BaseModel):
    description: str = Field(description="What is blocking progress.")
    raised_by: str = Field(description="Person or team who raised the blocker.")
    resolution_needed_from: str = Field(
        description="Who needs to act to resolve this blocker."
    )


class DeliverableOwner(BaseModel):
    deliverable: str = Field(description="Name of the deliverable.")
    owner: str = Field(description="Responsible person or team.")
    due_date: str = Field(description="Target date.")


class ProjectState(BaseModel):
    project_name: str = Field(description="Project or engagement name.")
    overall_status: Literal["green", "amber", "red"] = Field(
        description="RAG status: green = on track, amber = at risk, red = in trouble."
    )
    milestones: list[Milestone] = Field(description="Key milestones and their statuses.")
    risks: list[Risk] = Field(description="Active risks ranked by severity.")
    blockers: list[Blocker] = Field(description="Current blockers requiring resolution.")
    deliverable_owners: list[DeliverableOwner] = Field(
        description="RACI-style deliverable ownership list."
    )
    summary: str = Field(
        description="Two-to-three sentence plain-English summary of project health for an executive audience."
    )


class UpdateInput(BaseModel):
    source: str = Field(description="Who or what this update came from, e.g. 'weekly status call'.")
    content: str = Field(description="Raw unstructured update text.")


# ── Prompts ───────────────────────────────────────────────────────────────────

_EXTRACT_SYSTEM = SystemMessage(
    """You are a project management analyst. Read the update text and extract
any changes to milestones, risks, blockers, or deliverable ownership.

Return a ProjectState reflecting ONLY what was mentioned in this update.
Leave all other fields as empty lists. Set overall_status to the status
implied by the update, or 'green' if not mentioned."""
)

_MERGE_SYSTEM = SystemMessage(
    """You are a senior PMO director. You have the current project state and a
delta extracted from the latest update.

Merge the delta into the current state:
- Add new milestones, risks, blockers, and deliverable owners.
- Update existing items if the new information supersedes the old (match by name/description).
- Remove blockers marked as resolved.
- Re-derive the overall RAG status based on the merged picture.
- Write a fresh two-to-three sentence executive summary.

Return the complete merged ProjectState."""
)


# ── Engine ────────────────────────────────────────────────────────────────────

def init_state(project_name: str) -> ProjectState:
    """Return a blank project state for a new engagement."""
    return ProjectState(
        project_name=project_name,
        overall_status="green",
        milestones=[],
        risks=[],
        blockers=[],
        deliverable_owners=[],
        summary="Project initialised. No updates received yet.",
    )


def apply_update(current: ProjectState, update: UpdateInput) -> ProjectState:
    """Apply one unstructured update to the current project state."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    structured_llm = llm.with_structured_output(ProjectState)

    delta: ProjectState = structured_llm.invoke(
        [
            _EXTRACT_SYSTEM,
            {
                "role": "user",
                "content": (
                    f"Project: {current.project_name}\n"
                    f"Source: {update.source}\n"
                    f"Update:\n{update.content}"
                ),
            },
        ]
    )

    merged: ProjectState = structured_llm.invoke(
        [
            _MERGE_SYSTEM,
            {
                "role": "user",
                "content": (
                    f"Current state:\n{current.model_dump_json(indent=2)}\n\n"
                    f"Delta from update ({update.source}):\n{delta.model_dump_json(indent=2)}"
                ),
            },
        ]
    )
    return merged


def run(project_name: str, updates: list[UpdateInput]) -> list[ProjectState]:
    """Apply a sequence of updates and return the state after each one."""
    state = init_state(project_name)
    history = [state]
    for update in updates:
        state = apply_update(state, update)
        history.append(state)
    return history

## The scenario

A new ERP rollout is kicking off across three regional offices. Over eight weeks, the PMO receives a kick-off summary, a steering committee call flagging a vendor integration blocker, and a programme director email reporting a partial resolution with a revised timeline. The agent ingests each update in plain text and keeps the project record current — no spreadsheets, no manual updates.

In [ ]:
PROJECT = "ERP Rollout -- EMEA Region"

UPDATES = [
    UpdateInput(
        source="Kick-off meeting notes (Week 1)",
        content=(
            "ERP rollout kicked off across EMEA. Four milestones defined: "
            "legacy data cleanse by end of Q2-2025, system configuration sign-off by Q3-2025, "
            "parallel-run testing by Q4-2025, and go-live for all three offices by Q1-2026. "
            "Priya Nair leads data cleanse, Marco Rossi owns system configuration. "
            "No risks or blockers at this stage."
        ),
    ),
    UpdateInput(
        source="Steering committee call (Week 5)",
        content=(
            "Data cleanse is tracking well but a blocker has emerged: the ERP vendor has not "
            "provided the field-mapping specification needed for system configuration. "
            "Marco Rossi raised the blocker -- vendor management needs to escalate. "
            "Risk logged: if spec not received by end of month, Q3 configuration sign-off will slip. "
            "Severity is high. Overall project is amber."
        ),
    ),
    UpdateInput(
        source="Programme director email (Week 8)",
        content=(
            "Vendor escalation worked -- field-mapping spec received. Blocker is resolved. "
            "However, the configuration sign-off milestone has slipped four weeks to mid Q3-2025 "
            "due to lost time. Parallel-run testing pushed to early Q1-2026 as a result. "
            "Go-live is now at risk -- new provisional date is Q2-2026 pending testing outcomes. "
            "Project status remains amber. Priya Nair's data cleanse completed on time."
        ),
    ),
]

history = run(PROJECT, UPDATES)

STATUS_LABELS = {"green": "GREEN", "amber": "AMBER", "red": "RED"}

for i, state in enumerate(history):
    label = "INITIAL STATE" if i == 0 else f"After update {i}: {UPDATES[i-1].source}"
    print(f"\n{'=' * 64}")
    print(label)
    print(f"Project  : {state.project_name}")
    print(f"Status   : {STATUS_LABELS[state.overall_status]}")
    print(f"Summary  : {state.summary}")

    if state.milestones:
        print(f"\nMilestones ({len(state.milestones)}):")
        for m in state.milestones:
            print(f"  [{m.status}] {m.name} -- {m.due_date}")

    if state.blockers:
        print(f"\nBlockers ({len(state.blockers)}):")
        for b in state.blockers:
            print(f"  {b.description}")
            print(f"    Raised by: {b.raised_by}  |  Needs action from: {b.resolution_needed_from}")

    if state.risks:
        print(f"\nRisks ({len(state.risks)}):")
        for r in state.risks:
            print(f"  [{r.severity.upper()}] {r.description}")
            print(f"    Owner: {r.owner}")

    if state.deliverable_owners:
        print(f"\nOwnership:")
        for d in state.deliverable_owners:
            print(f"  {d.deliverable} -- {d.owner} (due {d.due_date})")

## Use your own data

Replace the input above with:
- `PROJECT` — your project or engagement name
- `UPDATES` — a list of `UpdateInput` objects, one per update (email, call notes, Slack thread, whatever you have)

The agent will return a full project record after each update: RAG status, milestone tracker, active risks with owners, and an executive summary ready to paste into a board report.